1. Importing Necessary Libraries

In [26]:
from datetime import datetime
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import os, re
from pathlib import Path
import numpy as np
import numpy as _np
import pandas as pd
import cv2
import tensorflow as tf
from tensorflow import keras
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix
)
print("TF:", tf.__version__)

TF: 2.20.0


2. Configuration 

In [ ]:
CONFIG = {
    # Models to evaluate (name -> file path) 
    "models": {
        "ANN": "models/fruit_quality_ann_detector.keras",
        "CNN": "models/fruit_quality_cnn_detector.keras",
        "RNN": "models/fruit_quality_rnn_detector.keras",
        "SEMI_CNN": "models/fruit_quality_model_v_semi.keras",
        "SEMI_CNN_LIFELONG": "models/fruit_quality_teacher_ep3.keras",
    },

    "annotation": {
        "type": "csv",  
        "path": r"C:/Users/evri/Desktop/Διπλωματικη/Adversarial AI Attack Detection/Adversarial-AI-Attack-Detection/fruit_quality_train_detector_annotations.csv",
        #"path": r"C:/Users/evri/Desktop/Διπλωματικη/Adversarial AI Attack Detection/Adversarial-AI-Attack-Detection/fruit_quality_unkonwn_adversarial_fgsm_RNN_detector_annotations.csv",
        #"path": r"C:/Users/evri/Desktop/Διπλωματικη/Adversarial AI Attack Detection/Adversarial-AI-Attack-Detection/fruit_quality_unkonwn_adversarial_PGD_detector_annotations.csv",
        "base_dir": "",   
        "filename_template": "{image_id}",  
        "label_preference": ["adv_label", "label"],     
        "test_value": "test",  
        "aux_columns": ["isNight"],   
    },
    "img_size": 100,        
    "num_channels": 3,     
    "batch_size": 64,      

    "out_dir": ".",
}
print("CONFIG loaded.")

CONFIG loaded.


3. Utility & Data Loading Helpers

In [35]:
def ensure_out_dir():
    p = Path(CONFIG.get("out_dir","eval_outputs"))
    p.mkdir(parents=True, exist_ok=True)
    return p

def _infer_col(df, choices):
    for c in choices:
        if c in df.columns: return c
    lowers = {c.lower(): c for c in df.columns}
    for c in choices:
        if c.lower() in lowers: return lowers[c.lower()]
    return None

def _infer_split_col(df):
    for c in ["split","subset","partition","set","split_type","phase","dataset","usage","fold"]:
        if c in df.columns: return c
    for c in df.columns:
        cl = c.lower()
        if "split" in cl or "subset" in cl or "partition" in cl:
            return c
    return None

def _to_int_label(v):
    try:
        if isinstance(v, str):
            s = v.strip().lower()
            if s in ("1","true","yes","attack","adv","adversarial","malicious"): return 1
            if s in ("0","false","no","clean","benign","normal"): return 0
            return int(float(s))
        return int(v)
    except Exception:
        return np.nan

def _to_float_aux(v):
    if isinstance(v, (int,float,_np.integer,_np.floating)): return float(v)
    if isinstance(v, (bool,_np.bool_)): return 1.0 if v else 0.0
    if isinstance(v,str):
        s=v.strip().lower()
        if s in ("1","true","yes","y"): return 1.0
        if s in ("0","false","no","n"): return 0.0
        try: return float(s)
        except: return 0.0
    try: return float(v)
    except: return 0.0

def _build_path_from_row(row, base_dir, tmpl):
    for c in ["path","filepath","file","image_path","image","filename"]:
        v = row.get(c, None)
        if isinstance(v,str) and v.strip(): return os.path.normpath(v)
    if tmpl and "{" in tmpl and "}" in tmpl:
        try:
            rel = tmpl.format(**row)
            return os.path.normpath(os.path.join(base_dir, rel) if base_dir else rel)
        except: pass
    if "image_id" in row and isinstance(row["image_id"], str):
        rel = row["image_id"]
        return os.path.normpath(os.path.join(base_dir, rel) if base_dir else rel)
    return None

def load_df_TEST_only(ann_cfg: dict) -> pd.DataFrame:
    assert ann_cfg.get("type","csv") == "csv", "Only CSV annotations supported here."
    df = pd.read_csv(ann_cfg["path"])
    split_col = _infer_split_col(df)
    tv = str(ann_cfg.get("test_value","test")).lower()
    if split_col is not None:
        df = df[df[split_col].astype(str).str.lower()==tv].copy()
    else:
        print("⚠️ No split column — using all rows as TEST.")

    label_col = None
    for c in ann_cfg.get("label_preference", ["label"]):
        if c in df.columns: label_col=c; break
    if label_col is None:
        label_col = _infer_col(df, ["label","adv_label","y","target"])
    if label_col is None:
        raise ValueError("Label column not found in annotation CSV.")
    
    df["__label"] = df[label_col].apply(_to_int_label).astype("float")
    df = df.dropna(subset=["__label"]).copy()
    df["__label"] = df["__label"].astype(int)

    rows = df.to_dict("records")
    base_dir = ann_cfg.get("base_dir","")
    tmpl = ann_cfg.get("filename_template","{image_id}")
    files = [_build_path_from_row(r, base_dir, tmpl) for r in rows]
    df["__file"] = files
    df = df[df["__file"].notna()].copy()
    return df

def imread_unicode(path: str, num_channels=3):
    data = np.fromfile(path, dtype=np.uint8)
    flag = cv2.IMREAD_COLOR if num_channels==3 else cv2.IMREAD_GRAYSCALE
    img = cv2.imdecode(data, flag)
    if img is None: raise FileNotFoundError(f"imdecode failed: {path}")
    return img

def preprocess_img(img, size, num_channels):
    if num_channels==3 and img.ndim==2: img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    if num_channels==1 and img.ndim==3: img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    img = cv2.resize(img, (size,size), interpolation=cv2.INTER_AREA)
    img = img.astype(np.float32)/255.0
    if num_channels==1 and img.ndim==2: img = img[...,None]
    return img

def build_inputs(files, df, size, num_channels, aux_cols):
    N=len(files); H=W=size; C=num_channels; k=len(aux_cols)
    X_img = np.zeros((N,H,W,C), dtype=np.float32)
    X_flat= np.zeros((N,H*W*C), dtype=np.float32)
    X_aux = np.zeros((N,k), dtype=np.float32) if k>0 else None
    rows = df.to_dict("records")
    for i,(p,r) in enumerate(zip(files, rows)):
        img = preprocess_img(imread_unicode(p, num_channels), size, num_channels)
        X_img[i]=img; X_flat[i]=img.reshape(-1)
        if k: X_aux[i]=[ _to_float_aux(r.get(c,0)) for c in aux_cols ]
    return X_img, X_flat, X_aux

def describe_model_inputs(model):
    ins = model.inputs if isinstance(model.inputs,(list,tuple)) else [model.input]
    specs=[]
    for t in ins:
        sh = tuple(int(d) if d is not None else None for d in t.shape)
        specs.append({"shape":sh, "rank":len(sh)})
    return specs

def _make_seq_from_image(X_img, target_T=None, target_F=None, H=None, W=None, C=None):
    N = X_img.shape[0]
    if H is None or W is None or C is None:
        H, W, C = X_img.shape[1:4]
    T = target_T if target_T is not None else H
    F = target_F if target_F is not None else (W*C)
    if H*W*C != T*F:
        return X_img.reshape(N, H*W*C)[:, :T*F].reshape(N, T, F)
    return X_img.reshape(N, H, W*C) if (T==H and F==W*C) else X_img.reshape(N, H*W*C).reshape(N, T, F)

def assemble_inputs(model, X_img, X_flat, X_aux):
    specs = describe_model_inputs(model)
    H,W,C = X_img.shape[1:4]; d_flat = X_flat.shape[1]; d_aux = 0 if X_aux is None else X_aux.shape[1]
    X_seq_default = X_img.reshape(X_img.shape[0], H, W*C) 

    if len(specs)==1:
        sp=specs[0]
        if sp["rank"]==4:
            return X_img
        if sp["rank"]==3:
            T = sp["shape"][1]
            F = sp["shape"][2]
            x_seq = _make_seq_from_image(X_img, target_T=T, target_F=F, H=H, W=W, C=C)
            return x_seq
        if sp["rank"]==2:
            dim=sp["shape"][-1]
            if dim is None:
                return np.concatenate([X_flat,X_aux],1) if d_aux>0 else X_flat
            if d_aux>0 and d_flat+d_aux==dim: return np.concatenate([X_flat,X_aux],1)
            if d_flat==dim: return X_flat
            if d_aux>0 and d_aux==dim: return X_aux
            return np.concatenate([X_flat,X_aux],1) if d_aux>0 else X_flat
        raise ValueError(f"Unsupported input rank: {sp}")
    elif len(specs)==2:
        x_list=[None,None]
        for i,sp in enumerate(specs):
            if sp["rank"]==4:
                x_list[i]=X_img
            elif sp["rank"]==3:
                T = sp["shape"][1]
                F = sp["shape"][2]
                x_list[i]= _make_seq_from_image(X_img, target_T=T, target_F=F, H=H, W=W, C=C)
            elif sp["rank"]==2:
                dim=sp["shape"][-1]
                if dim is None:
                    x_list[i]=X_aux if (X_aux is not None) else X_flat
                elif X_aux is not None and dim==d_aux:
                    x_list[i]=X_aux
                elif dim==d_flat:
                    x_list[i]=X_flat
                elif X_aux is not None and dim==(d_flat+d_aux):
                    x_list[i]=np.concatenate([X_flat,X_aux],1)
                else:
                    x_list[i]=X_flat  # fallback
            else:
                raise ValueError(f"Unsupported rank in multi-input: {sp}")
        return x_list
    else:
        raise ValueError(f"Model with {len(specs)} inputs is not supported here.")

class BinaryEvalExcel:
    def __init__(self, class_names=("clean","attack")):
        self.class_names=list(class_names)
        self.sum_rows=[]; self.tpfp_rows=[]; self.conf_rows=[]

    def add(self, model, y_true, y_scores=None, y_pred=None, threshold=0.5):
        y_true=np.asarray(y_true).astype(int).ravel()
        if y_scores is None and y_pred is None: raise ValueError("Provide y_scores or y_pred")
        if y_scores is not None:
            y_scores=np.asarray(y_scores).ravel()
            if y_pred is None: y_pred=(y_scores>=threshold).astype(int)
        else:
            y_pred=np.asarray(y_pred).astype(int).ravel()
        tn,fp,fn,tp = confusion_matrix(y_true,y_pred,labels=[0,1]).ravel()
        self.sum_rows.append({
            "model":model,"n_samples":int(len(y_true)),
            "accuracy":float(accuracy_score(y_true,y_pred)),
            "precision":float(precision_score(y_true,y_pred,pos_label=1,zero_division=0)),
            "recall":float(recall_score(y_true,y_pred,pos_label=1,zero_division=0)),
            "f1":float(f1_score(y_true,y_pred,pos_label=1,zero_division=0)),
            "roc_auc":float(roc_auc_score(y_true,y_scores)) if y_scores is not None else float("nan"),
            "pr_auc":float(average_precision_score(y_true,y_scores)) if y_scores is not None else float("nan"),
        })
        self.tpfp_rows.append({"model":model,"TP":int(tp),"TN":int(tn),"FP":int(fp),"FN":int(fn)})

        cm=np.array([[tn,fp],[fn,tp]],dtype=int)  
        for i_true in (0,1):
            for j_pred in (0,1):
                self.conf_rows.append({"model":model,"true":self.class_names[i_true],"pred":self.class_names[j_pred],"count":int(cm[i_true,j_pred])})
    
    def save_excel(self, path):
        df1=pd.DataFrame(self.sum_rows)[["model","n_samples","accuracy","precision","recall","f1","roc_auc","pr_auc"]]
        df2=pd.DataFrame(self.tpfp_rows)[["model","TP","TN","FP","FN"]]
        df3=pd.DataFrame(self.conf_rows)[["model","true","pred","count"]]
        path=Path(path); path.parent.mkdir(parents=True,exist_ok=True)
        with pd.ExcelWriter(path) as xw:
            df1.to_excel(xw, sheet_name="Summary", index=False)
            df2.to_excel(xw, sheet_name="TP_TN_FP_FN", index=False)
            df3.to_excel(xw, sheet_name="Confusions", index=False)
        display(df1); print(f"✓ Saved:", path)

4. Run Evaluation & Export (Excel)

In [37]:
out_dir = ensure_out_dir()
df = load_df_TEST_only(CONFIG["annotation"])
files = df["__file"].tolist()
y = df["__label"].to_numpy().astype(int)
aux_cols = CONFIG["annotation"].get("aux_columns", []) or []
print(f"TEST samples: {len(y)}  |  Aux columns: {aux_cols}")

X_img, X_flat, X_aux = build_inputs(
    files, df,
    size=int(CONFIG["img_size"]),
    num_channels=int(CONFIG["num_channels"]),
    aux_cols=aux_cols
)
print("Inputs:", "X_img", X_img.shape, "| X_flat", X_flat.shape, "| X_aux", None if X_aux is None else X_aux.shape)

models = {}
for name, path in CONFIG["models"].items():
    if Path(path).exists():
        try:
            models[name] = keras.models.load_model(path)
            print(f"✓ Loaded {name} from {path}")
            print("  Input specs:", describe_model_inputs(models[name]))
        except Exception as e:
            print(f"⚠️ Could not load {name}: {e}")
    else:
        print(f"⚠️ Missing model file: {path}")

agg = BinaryEvalExcel(class_names=("clean","attack"))
for name, model in models.items():
    print(f"\n=== Evaluating {name} ===")
    model_input = assemble_inputs(model, X_img, X_flat, X_aux)

    scores = model.predict(model_input, verbose=0).squeeze()
    if scores.ndim > 1:
        if scores.shape[-1] == 1:
            scores = scores[..., 0]
        else:
            scores = scores[..., -1]

    y_pred = (scores >= 0.5).astype(int)

    agg.add(model=name, y_true=y, y_scores=scores, y_pred=y_pred)

excel_path = Path(out_dir) / "evaluation_fruit_quality_detector_summary.xlsx"
#excel_path = Path(out_dir) / "evaluation_fruit_quality_detector_unknown_fgsm_RNN_summary.xlsx" 
#excel_path = Path(out_dir) / "evaluation_fruit_quality_detector_unknown_PGD_summary.xlsx"
agg.save_excel(excel_path)

TEST samples: 7812  |  Aux columns: ['isNight']
Inputs: X_img (7812, 100, 100, 3) | X_flat (7812, 30000) | X_aux (7812, 1)
✓ Loaded ANN from models/fruit_quality_ann_detector.keras
  Input specs: [{'shape': (None, 30001), 'rank': 2}]
✓ Loaded CNN from models/fruit_quality_cnn_detector.keras
  Input specs: [{'shape': (None, 100, 100, 3), 'rank': 4}, {'shape': (None, 1), 'rank': 2}]
✓ Loaded RNN from models/fruit_quality_rnn_detector.keras
  Input specs: [{'shape': (None, 100, 300), 'rank': 3}, {'shape': (None, 1), 'rank': 2}]

=== Evaluating ANN ===

=== Evaluating CNN ===

=== Evaluating RNN ===


,model,n_samples,accuracy,precision,recall,f1,roc_auc,pr_auc
0,ANN,7812,0.455453,0.976827,0.280594,0.435959,0.731478,0.900030
1,CNN,7812,0.530338,0.987100,0.378734,0.547428,0.882056,0.955941
2,RNN,7812,0.392345,0.971186,0.195597,0.325614,0.774598,0.904082


✓ Saved: evaluation_fruit_quality_detector_unknown_PGD_summary.xlsx


In [ ]:
OUT_DIR = Path("./fruit_quality_eval_detector_viz")                     
#OUT_DIR = Path("./fruit_quality_eval_detector_unknown_fgsm_RNN_viz")     
#OUT_DIR = Path("./fruit_quality_eval_detector_unknown_PGD_viz")      
FILE_PREFIX = "eval"                                       
MAKE_PDF = True                                             

OUT_DIR.mkdir(parents=True, exist_ok=True)

__fig_counter = {'i': 0}    
__orig_show = plt.show     
__pdf = {'obj': PdfPages(OUT_DIR / f"evaluation_visual_report_{datetime.now().strftime('%Y%m%d-%H%M%S')}.pdf")} if MAKE_PDF else {'obj': None}

def _save_current_fig():
    fignums = plt.get_fignums()
    if not fignums:
        return
    num = fignums[-1]           
    fig = plt.figure(num)  

    __fig_counter['i'] += 1
    fname = f"{FILE_PREFIX}_{__fig_counter['i']:02d}.png"
    fig.savefig(OUT_DIR / fname, dpi=150, bbox_inches="tight")

    if __pdf['obj'] is not None:
        __pdf['obj'].savefig(fig)

def _show_and_save(*args, **kwargs):
    _save_current_fig()
    return __orig_show(*args, **kwargs)

plt.show = _show_and_save

def finalize_autosave():
    if __pdf['obj'] is not None:
        try:
            __pdf['obj'].close()
            print("PDF saved.")
        except Exception as e:
            print("⚠ Error while closing PDF:", e)
    print(f"[AutoSave] Saved {__fig_counter['i']} PNG(s)  to:", OUT_DIR)

print(f"[AutoSave] Enabled. PNGs => {OUT_DIR}, PDF => {MAKE_PDF}")

In [ ]:
CLASS_NAMES = ['clean', 'adversarial']

DEFAULT_SUMMARY = Path('.') / 'evaluation_fruit_quality_detector_summary.xlsx'
#DEFAULT_SUMMARY = Path('.') / 'evaluation_fruit_quality_detector_unknown_fgsm_RNN_summary.xlsx'
#DEFAULT_SUMMARY = Path('.') / 'evaluation_fruit_quality_detector_unknown_PGD_summary.xlsx'
if DEFAULT_SUMMARY.exists():
    SUMMARY_PATH = DEFAULT_SUMMARY
else:
    candidates = list(Path('.').glob('evaluation_fruit_quality_detector_summary.xlsx'))
    #candidates = list(Path('.').glob('evaluation_fruit_quality_detector_unknown_fgsm_RNN_summary.xlsx'))
    #candidates = list(Path('.').glob('evaluation_fruit_quality_detector_unknown_PGD_summary.xlsx'))

    SUMMARY_PATH = max(candidates, key=lambda p: p.stat().st_mtime) if candidates else Path(r'C:/Users/evri/Desktop/Διπλωματικη/Adversarial AI Attack Detection/Adversarial-AI-Attack-Detection/evaluation_fruit_quality_detector_summary.xlsx')
    #SUMMARY_PATH = max(candidates, key=lambda p: p.stat().st_mtime) if candidates else Path(r'C:/Users/evri/Desktop/Διπλωματικη/Adversarial AI Attack Detection/Adversarial-AI-Attack-Detection/evaluation_fruit_quality_detector_unknown_fgsm_RNN_summary.xlsx')
    #SUMMARY_PATH = max(candidates, key=lambda p: p.stat().st_mtime) if candidates else Path(r'C:/Users/evri/Desktop/Διπλωματικη/Adversarial AI Attack Detection/Adversarial-AI-Attack-Detection/evaluation_fruit_quality_detector_unknown_PGD_summary.xlsx')

print('SUMMARY_PATH:', SUMMARY_PATH)
print('CLASS_NAMES:', CLASS_NAMES)

In [ ]:
def read_summary_sheets(xlsx_path: Path):
    try:
        df_summary = pd.read_excel(xlsx_path, sheet_name='Summary', engine='openpyxl')
    except Exception:
        df_summary = pd.read_excel(xlsx_path, sheet_name='Summary')

    try:
        df_tpfp = pd.read_excel(xlsx_path, sheet_name='TP_TN_FP_FN', engine='openpyxl')
    except Exception:
        try:
            df_tpfp = pd.read_excel(xlsx_path, sheet_name='TP_TN_FP_FN')
        except Exception:
            df_tpfp = None

    try:
        df_cm = pd.read_excel(xlsx_path, sheet_name='Confusions', engine='openpyxl')
    except Exception:
        try:
            df_cm = pd.read_excel(xlsx_path, sheet_name='Confusions')
        except Exception:
            df_cm = None

    return df_summary, df_tpfp, df_cm

df_summary, df_tpfp, df_cm = read_summary_sheets(SUMMARY_PATH)

print('Summary head:'); 
display(df_summary.head())

print("TP_TN_FP_FN head:" if df_tpfp is not None else "No 'TP_TN_FP_FN' sheet found.")
display(df_tpfp.head()) if df_tpfp is not None else None

print('Confusions head:' if df_cm is not None else "No 'Confusions' sheet found (optional).")
display(df_cm.head()) if df_cm is not None else None

In [ ]:
pv_acc = (
    df_summary
    .groupby('model', dropna=False)['accuracy']
    .mean()                        
    .to_frame(name='accuracy')
    .sort_index()
)
display(pv_acc.style.format('{:.4f}').set_caption('Accuracy by Model'))

pv_precision = (
    df_summary
    .groupby('model', dropna=False)['precision']
    .mean()
    .to_frame(name='precision')
    .sort_index()
)
display(pv_precision.style.format('{:.4f}').set_caption('precision by Model'))

pv_recall = (
    df_summary
    .groupby('model', dropna=False)['recall']
    .mean()
    .to_frame(name='recall')
    .sort_index()
)
display(pv_recall.style.format('{:.4f}').set_caption('recall by Model'))

In [ ]:
def grouped_bars(metric_col: str, title: str, df_src=None):
    df = (df_src if df_src is not None else df_summary).copy()

    df.columns = [str(c).strip().lower() for c in df.columns]

    if 'model' not in df.columns and ('model' in (df.index.names or [])):
        df = df.reset_index()
    if 'model' not in df.columns:
        raise KeyError("Missing 'model' column or index.")

    ds_col = next((c for c in ['dataset','task','domain','set','corpus','source'] if c in df.columns), None)
    if ds_col is None:
        df['dataset'] = 'all'
        ds_col = 'dataset'

    aliases = {
        'accuracy': ['accuracy','acc','top1','top_1','acc1','top1_acc','top-1'],
        'precision': ['precision','prec','ppv'],
        'recall': ['recall','tpr','sensitivity','rec'],
        'f1': ['f1','f1_score','f1-score'],
        'roc_auc': ['roc_auc','auroc','roc-auc','auc_roc','rocauc'],
        'pr_auc': ['pr_auc','average_precision','aupr','ap','pr-auc','avg_precision'],
    }
    candidate_cols = aliases.get(metric_col, [metric_col])
    mcol = next((c for c in candidate_cols if c in df.columns), None)
    if mcol is None:
        print(f"⚠️ No column found for '{metric_col}'. Available: {list(df.columns)}")
        return

    df[mcol] = pd.to_numeric(df[mcol], errors='coerce')
    df = (
        df.groupby(['model', ds_col], dropna=False, as_index=False)[mcol]
          .mean()
    )

    models = sorted(df['model'].astype(str).unique())
    datasets = sorted(df[ds_col].astype(str).unique())

    width = 0.8 / max(len(models), 1) 
    x = np.arange(len(datasets))
    plt.figure(figsize=(max(10, 1.4 * len(datasets)), 4))

    for i, m in enumerate(models):
        sub = df[df['model'] == m].set_index(ds_col).reindex(datasets)
        vals = sub[mcol].values
        plt.bar(x + i * width, vals, width, label=m)

    plt.xticks(x + width * (len(models) - 1) / 2, datasets, rotation=30, ha='right')
    plt.ylabel(metric_col)
    plt.title(title)
    plt.legend(frameon=False)
    plt.tight_layout()
    plt.show()

grouped_bars('accuracy',  'Accuracy — models grouped within each dataset')
grouped_bars('precision', 'Precision — models grouped within each dataset')
grouped_bars('recall',    'Recall — models grouped within each dataset')
grouped_bars('f1',        'F1 — models grouped within each dataset')
grouped_bars('roc_auc',   'ROC AUC — models grouped within each dataset')
grouped_bars('pr_auc',    'PR AUC — models grouped within each dataset')

In [ ]:
_df_tpfp = df_tpfp.copy()
_df_tpfp.columns = [str(c).strip().lower() for c in _df_tpfp.columns]

required = {'model', 'tp', 'tn', 'fp', 'fn'}
missing = required - set(_df_tpfp.columns)
if missing:
    raise KeyError(f"Missing columns in df_tpfp: {missing}. Got: {list(_df_tpfp.columns)}")

if 'dataset' not in _df_tpfp.columns:
    _df_tpfp['dataset'] = 'all'

_df_tpfp = (
    _df_tpfp
    .groupby(['model', 'dataset'], as_index=False)[['tp','tn','fp','fn']]
    .sum()
)

def plot_tpfp_for(model: str, dataset: str):
    sub = _df_tpfp[(_df_tpfp['model'].astype(str) == str(model)) &
                   (_df_tpfp['dataset'].astype(str) == str(dataset))]
    if sub.empty:
        print(f"⚠️ No rows for model={model}, dataset={dataset}")
        return

    row = sub[['tp','tn','fp','fn']].sum()
    counts = row.rename(index={'tp':'TP','tn':'TN','fp':'FP','fn':'FN'})

    counts.to_frame(name='count').T.plot(
        kind='bar', stacked=True, figsize=(7, 4),
        title=f'TP/FP/FN/TN — {model} @ {dataset}', legend=True
    )
    plt.ylabel('Count')
    plt.tight_layout()
    plt.show()

combos = _df_tpfp[['model','dataset']].drop_duplicates().values
for model, dataset in combos:
    plot_tpfp_for(model, dataset)

In [ ]:
_cm = df_cm.copy()
_cm.columns = [str(c).strip().lower() for c in _cm.columns]

required = {'model', 'true', 'pred', 'count'}
missing = required - set(_cm.columns)
if missing:
    raise KeyError(f"Missing columns in df_cm: {missing}. Got: {list(_cm.columns)}")

if 'dataset' not in _cm.columns:
    _cm['dataset'] = 'all'

_cm['count'] = pd.to_numeric(_cm['count'], errors='coerce').fillna(0).astype(int)
_cm['true']  = _cm['true'].astype(str)
_cm['pred']  = _cm['pred'].astype(str)

preferred = ['clean', 'attack']
unique_classes = sorted(set(_cm['true']).union(set(_cm['pred'])), key=str)
order = preferred if all(c in unique_classes for c in preferred) else unique_classes

def plot_cm_for(model: str, dataset: str, order=order):
    sub = _cm[
        (_cm['model'].astype(str) == str(model)) &
        (_cm['dataset'].astype(str) == str(dataset))
    ]
    if sub.empty:
        print(f"⚠️ No rows for model={model}, dataset={dataset}")
        return

    mat_df = (
        sub.pivot_table(index='true', columns='pred', values='count', aggfunc='sum')
           .reindex(index=order, columns=order, fill_value=0)
    )
    mat = mat_df.to_numpy()

    fig, ax = plt.subplots(figsize=(4.8, 4.2))
    im = ax.imshow(mat, interpolation='nearest')
    ax.set_xticks(range(len(order))); ax.set_yticks(range(len(order)))
    ax.set_xticklabels(order);        ax.set_yticklabels(order)
    ax.set_xlabel('Predicted');       ax.set_ylabel('True')
    ax.set_title(f'Confusion — {model} @ {dataset}')

    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            ax.text(j, i, int(mat[i, j]), ha='center', va='center')

    plt.tight_layout()
    plt.show()

for model, dataset in _cm[['model', 'dataset']].drop_duplicates().values:
    plot_cm_for(model, dataset, order=order)

In [ ]:
try:
    finalize_autosave()
except NameError:
    print("AutoSave dont run.")